# A1.4 · Designing the agent control plane

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Security of AI*

Builds on **[A1.3 · Architecture review when the system acts](https://spbreed.github.io/cyber-commons/lessons/A1.3.html)**.

| | |
|---|---|
| Open-source tooling | kind, SPIRE, agentgateway, OPA |
| Open-weight models | Llama 3.3 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


A1.3 established that the control plane is where a design decision can actually
bind. This lesson builds it.

A control plane for an agent has exactly three levers, and they map onto
questions an attacker asks:

| Lever | The attacker's question | Real-world equivalent |
|---|---|---|
| **Tool policy** | what can I invoke? | OPA / Kyverno admission policy |
| **Egress policy** | who can I talk to? | Cilium network policy, a proxy allowlist |
| **Path policy** | what can I read or write? | container mounts, seccomp, AppArmor |

Each one must be **deny-by-default**. That is not paranoia — it is the only
setting that survives someone adding a capability and forgetting to update the
policy, which is the normal case, not the exception.

And every decision must return a **reason**. A control you cannot explain is a
control you cannot tune, and an untunable control gets switched off the first
time it blocks something legitimate. That is how security controls actually die
— not overridden, just quietly disabled by a tired engineer at 6pm.

## 2 · Demo — the three levers, deny-by-default

This is a working control plane in about sixty lines. In production the same three decisions come from OPA, Cilium and your container runtime; the logic is what matters here.

In [ ]:
import fnmatch, re
from dataclasses import dataclass, field
from urllib.parse import urlparse

@dataclass
class Decision:
    allowed: bool
    reason: str
    subject: str = ""
    def __str__(self):
        return f"{'ALLOW' if self.allowed else 'DENY ':5s} {self.subject:42s} {self.reason}"

PRIVATE = [re.compile(p) for p in (
    r"^127\.", r"^10\.", r"^192\.168\.", r"^169\.254\.",
    r"^172\.(1[6-9]|2\d|3[01])\.", r"^localhost$")]

@dataclass
class EgressPolicy:
    allow_hosts: set = field(default_factory=set)
    def check(self, url):
        host = (urlparse(url).hostname or "").lower()
        if not host:
            return Decision(False, "unparseable destination", url)
        if any(p.match(host) for p in PRIVATE):
            extra = " — cloud metadata service" if host.startswith("169.254") else ""
            return Decision(False, f"private/link-local address blocked{extra}", url)
        if host in self.allow_hosts:
            return Decision(True, "host on the allowlist", url)
        return Decision(False, "not on the egress allowlist (deny by default)", url)

@dataclass
class PathGuard:
    workspace: str = "/work"
    deny_globs: tuple = ("*/.ssh/*", "*/.aws/*", "*.pem", "*/.env", "*/etc/shadow")
    @staticmethod
    def normalise(path):
        parts = []
        for seg in path.split("/"):
            if seg in ("", "."):      continue
            if seg == "..":
                if parts: parts.pop()
                continue
            parts.append(seg)
        return "/" + "/".join(parts)
    def check(self, path):
        real = self.normalise(path)          # normalise BEFORE checking — see A3.3
        for g in self.deny_globs:
            if fnmatch.fnmatch(real, g):
                return Decision(False, f"matches deny rule {g}", path)
        ws = self.normalise(self.workspace)
        if real == ws or real.startswith(ws + "/"):
            return Decision(True, f"inside workspace ({real})", path)
        return Decision(False, f"outside workspace; resolves to {real}", path)

@dataclass
class ToolPolicy:
    allow: set = field(default_factory=set)
    require_approval: set = field(default_factory=set)
    deny: set = field(default_factory=set)
    def check(self, tool, approved=False):
        if tool in self.deny:
            return Decision(False, "tool explicitly denied", tool)
        if tool in self.require_approval and not approved:
            return Decision(False, "requires human approval, none presented", tool)
        if tool in self.allow or tool in self.require_approval:
            return Decision(True, "approved call" if approved else "on the allowlist", tool)
        return Decision(False, "not on the tool allowlist (deny by default)", tool)

print("three policies, defined. Nothing is permitted that was not named.")

## 3 · Demo — run a real agent session through it

These are the calls a code-review agent actually makes during one task, plus the three an attacker would make if the prompt were compromised. The control plane cannot tell the difference — and does not need to.

In [ ]:
@dataclass
class ControlPlane:
    egress: EgressPolicy
    paths: PathGuard
    tools: ToolPolicy
    log: list = field(default_factory=list)

    def call(self, tool, target="", approved=False):
        d = self.tools.check(tool, approved)
        if d.allowed:
            if target.startswith(("http://", "https://")):
                d = self.egress.check(target)
            elif target.startswith("/"):
                d = self.paths.check(target)
        self.log.append(d)
        return d

cp = ControlPlane(
    egress=EgressPolicy(allow_hosts={"api.github.com"}),
    paths=PathGuard(workspace="/work/repo"),
    tools=ToolPolicy(allow={"read_file", "search_code", "http_get"},
                     require_approval={"post_comment", "open_pr"},
                     deny={"merge_pr", "rotate_credential"}))

session = [
    # --- what the agent legitimately does ---
    ("read_file",   "/work/repo/src/auth.py",              False),
    ("search_code", "",                                     False),
    ("http_get",    "https://api.github.com/repos/x/y/pulls", False),
    ("post_comment", "",                                    False),   # ungated attempt
    ("post_comment", "",                                    True),    # with approval
    # --- what a compromised prompt would try ---
    ("read_file",   "/work/repo/../../root/.aws/credentials", False),
    ("http_get",    "http://169.254.169.254/latest/meta-data/iam/", False),
    ("http_get",    "https://exfil.example.com/collect",   False),
    ("rotate_credential", "",                               True),
]
for tool, target, approved in session:
    print(cp.call(tool, target, approved))

denied = [d for d in cp.log if not d.allowed]
print(f"\n{len(cp.log)} calls · {len(cp.log)-len(denied)} allowed · {len(denied)} denied")

## 4 · Where it breaks

Notice the last line: `rotate_credential` was refused **even though approval was presented**. That is deliberate, and it is the design decision this lesson exists to make.

An approval gate is only as good as the approver. Approve-every-call (L2) sounds strong until you measure the median approval latency — when it drops under two seconds, nobody is reading them, and the gate has become a click-through. For actions that are org-wide and irreversible, the right control is not a gate but a **denial**: that tool does not belong to this agent at all, and a different, narrower agent owns it.

This is why `deny` exists as a separate lever from `require_approval`.

In [ ]:
# Model the approval gate honestly: an approver under load.
def gate_effectiveness(calls_per_hour, seconds_per_real_review=45):
    seconds_available = 3600
    reviewable = seconds_available / seconds_per_real_review
    return {"calls": calls_per_hour, "can_truly_review": int(reviewable),
            "rubber_stamped": max(0, calls_per_hour - int(reviewable)),
            "fraction_real": round(min(reviewable / calls_per_hour, 1.0), 3)}

for rate in (10, 80, 400):
    r = gate_effectiveness(rate)
    print(f"{r['calls']:>4} approvals/hour → {r['fraction_real']:.0%} genuinely "
          f"reviewed, {r['rubber_stamped']:>3} rubber-stamped")
print("\nAt 400/hour the gate is theatre. Deny the irreversible tools instead,")
print("and give them to a separate agent with its own, much narrower manifest.")

## What you just proved

The legitimate calls succeed; `post_comment` is refused until approval is presented. The traversal, the metadata address and the exfiltration host are all denied with distinct reasons, and `rotate_credential` is denied despite approval. The gate model shows review quality collapsing to 20% at 400 approvals per hour.

## Your turn

Take the three levers and decide where each belongs in your stack: in the agent process, in a sidecar, or in the network. Only one of those placements still works when the agent process itself is the compromised component.

---

**Next → [A1.5 · Blast radius as a design metric](https://spbreed.github.io/cyber-commons/lessons/A1.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*